# 05. Pilot SNPE Ablation

此 notebook 只对 04 hard parity gate 通过的 schema 训练。公平 ablation 需要相同 prior、theta bank、split、simulation budget、seed protocol、architecture、normalization 和 early-stopping policy。

若没有 schema 通过 gate，此 notebook 会导出 explicit blocked manifest 和 non-object NPZ split metadata；它不会训练 NPE，不会生成 posterior，也不会制作伪造 recovery 或 PPC。

In [1]:
from __future__ import annotations
import json
import os
from pathlib import Path
import sys
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while not ((PROJECT_ROOT / ".git").exists() and (PROJECT_ROOT / "S4_sbi").exists()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate repository root")
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "S4_sbi" / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from sleep_sbi import overnight_ablation as oa

print("project root:", PROJECT_ROOT)
print("CONDA_DEFAULT_ENV:", os.environ.get("CONDA_DEFAULT_ENV"))
print("sys.executable:", sys.executable)
print("sys.prefix:", sys.prefix)
print("Python:", sys.version)
print("workflow version:", oa.SCHEMA_VERSION)
assert os.environ.get("CONDA_DEFAULT_ENV") == "neurolib"
assert "neurolib" in sys.executable.lower()
assert "neurolib" in sys.prefix.lower()

project root: D:\Year3_Mao_Projects\sleep_loop
CONDA_DEFAULT_ENV: neurolib
sys.executable: C:\Users\YUS190\AppData\Local\anaconda3\envs\neurolib\python.exe
sys.prefix: C:\Users\YUS190\AppData\Local\anaconda3\envs\neurolib
Python: 3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:36:49) [MSC v.1944 64 bit (AMD64)]
workflow version: overnight-observation-ablation-v0.1


## Gate before training

这里先检查 04 的 artifact。旧 5D/7D rate-summary banks 不能被重命名为新 EEG observation schema。

In [2]:
pilot_result = oa.run_pilot_ablation()
display(pilot_result["training"])
display(pilot_result["blockers"])
print(json.dumps(pilot_result["manifest"], indent=2))
assert not pilot_result["manifest"]["training_started"]
assert pilot_result["manifest"]["simulation_count"] == 0

,schema,dimension,training_status,train_simulations,validation_simulations,test_simulations,seeds_completed,runtime_s,reason
0,A_baseline14,14,Blocked by parity,0,0,0,0,0.0,"No fixed-length, same-semantic simulated EEG o..."
1,B_baseline_plus_so_morphology,16,Blocked by parity,0,0,0,0,0.0,"No fixed-length, same-semantic simulated EEG o..."
2,C_baseline_plus_spindle_occupancy,15,Blocked by parity,0,0,0,0,0.0,"No fixed-length, same-semantic simulated EEG o..."
3,D_baseline_plus_event_phase,17,Blocked by parity,0,0,0,0,0.0,"No fixed-length, same-semantic simulated EEG o..."
4,E_recommended_frozen_augmented,not frozen,Blocked by parity,0,0,0,0,0.0,"No fixed-length, same-semantic simulated EEG o..."
5,F_complete23_diagnostic,23,Blocked by parity,0,0,0,0,0.0,"No fixed-length, same-semantic simulated EEG o..."


,blocker,severity,evidence,minimum_recovery_step
0,extractor_semantic_parity,hard_gate,"No fixed-length, same-semantic simulated EEG o...",Implement and validate a simulator-to-observab...
1,frozen_observation_contract,hard_gate,No candidate has frozen simulator-side scaling...,Pre-register one fixed schema and retain indep...


{
  "schema_version": "overnight-observation-ablation-v0.1",
  "created_utc": "2026-07-27T01:46:30.882823+00:00",
  "training_started": false,
  "approved_schemas": [],
  "simulation_count": 0,
  "seeds_completed": 0,
  "status": "BLOCKED_BY_PARITY",
  "reason": "No fixed-length, same-semantic simulated EEG observable schema exists. Existing banks are legacy 5D/7D rate-summary banks and cannot be relabelled.",
  "scientific_boundary": "No posterior, posterior samples, recovery, calibration, or PPC results were generated because the simulation observations are semantically incompatible with the real EEG schemas."
}


## Artifact integrity and interpretation

空 split indices 是 hard gate 的真实记录，不是训练遗漏。没有 synthetic validation 时，任何真实 EEG posterior 都不应被声称为 parameter correctness evidence。

In [3]:
split_path = pilot_result["output_dir"] / "simulation_split_metadata.npz"
with np.load(split_path, allow_pickle=False) as archive:
    print([(key, archive[key].shape, str(archive[key].dtype)) for key in archive.files])
    assert not any(archive[key].dtype == object for key in archive.files)
print("No SNPE training was started because semantic parity did not pass.")

[('approved_schema_names', (0,), '<U1'), ('train_indices', (0,), 'int64'), ('validation_indices', (0,), 'int64'), ('test_indices', (0,), 'int64'), ('random_seeds', (0,), 'int64'), ('parity_status', (), '<U5'), ('reason', (), '<U512')]
No SNPE training was started because semantic parity did not pass.
